In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pyspark import SparkConf
from pyspark.sql import SparkSession
import pyspark.pandas as ps
import pyspark.sql.functions as F
import os

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(


In [4]:
from hypex.matching import Matching
from hypex.ml.faiss import FaissNearestNeighbors
from hypex.transformers import TypeCaster
from hypex.dataset import Dataset, InfoRole, TreatmentRole, FeatureRole, TargetRole, ExperimentData, AdditionalMatchingRole
from hypex.utils import BackendsEnum
from hypex.experiments import Experiment, OnRoleExperiment
from hypex.comparators import MahalanobisDistance
from hypex.encoders.encoders import DummyEncoder
from hypex.comparators import TTest, Chi2Test
from hypex.comparators.distances import MahalanobisDistance
from hypex.operators import Bias, MatchingMetrics
from hypex.analyzers import MatchingAnalyzer
from hypex.utils import SparkSessionCalculator
from hypex.config import DatasetConfig

In [17]:
path = "/home/eric/dataset/IHDP/csv"
files = os.listdir(path)
# with open(path + columns_file) as f:
#     print(f.read())
columns = (
    ["treatment", "y_factual", "y_cfactual", "mu0", "mu1"] +
    [f"x{i}" for i in range(1, 26)]
)
df = pd.DataFrame([], columns=columns)
# len(files)
for file in files:
    df = pd.concat(
        [
            df,
            pd.read_csv(path + "/" + files[0], names=columns, dtype=np.float64)
        ]
    )
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7470 entries, 0 to 746
Data columns (total 30 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   treatment   7470 non-null   float64
 1   y_factual   7470 non-null   float64
 2   y_cfactual  7470 non-null   float64
 3   mu0         7470 non-null   float64
 4   mu1         7470 non-null   float64
 5   x1          7470 non-null   float64
 6   x2          7470 non-null   float64
 7   x3          7470 non-null   float64
 8   x4          7470 non-null   float64
 9   x5          7470 non-null   float64
 10  x6          7470 non-null   float64
 11  x7          7470 non-null   float64
 12  x8          7470 non-null   float64
 13  x9          7470 non-null   float64
 14  x10         7470 non-null   float64
 15  x11         7470 non-null   float64
 16  x12         7470 non-null   float64
 17  x13         7470 non-null   float64
 18  x14         7470 non-null   float64
 19  x15         7470 non-null   float

In [18]:
df.head()

,treatment,y_factual,y_cfactual,mu0,mu1,x1,x2,x3,x4,x5,...,x16,x17,x18,x19,x20,x21,x22,x23,x24,x25
0,1.0,49.647921,34.950762,37.173291,50.383798,-0.528603,-0.343455,1.128554,0.161703,-0.316603,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,16.073412,49.435313,16.087249,49.546234,-1.736945,-1.802002,0.383828,2.244320,-0.629189,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,19.643007,48.598210,18.044855,49.661068,-0.807451,-0.202946,-0.360898,-0.879606,0.808706,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,26.368322,49.715204,24.605964,49.971196,0.390083,0.596582,-1.850350,-0.879606,-0.004017,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,20.258893,51.147418,20.612816,49.794120,-1.045229,-0.602710,0.011465,0.161703,0.683672,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
from hypex.utils.spark_config import SparkSessionCalculator

calculator = SparkSessionCalculator(
    num_rows=100_000_000,
    num_columns=7,
    num_categorical_columns=2,     # скорректируйте под ваши данные
    target_executor_memory_gb=10.0,
    target_executor_cores=4,
)

# 1. Рассчитать оптимальные настройки
settings = calculator.calculate_optimal_settings()

# 2. Оптимизировать существующую конфигурацию (с логированием изменений)
from pyspark import SparkConf
conf = SparkConf().setAppName("hypex_test").setMaster("k8s://...")
optimized_conf = calculator.optimize_config(conf)

# # 3. Создать сессию с оптимальными настройками
# sp_s = SparkSession.builder.config(conf=optimized_conf).getOrCreate()


SPARK SESSION CONFIGURATION OPTIMIZATION LOG

📊 TOTAL CHANGES: 15
   ➕ Added: 15
   🔄 Changed: 0

--------------------------------------------------------------------------------

➕ [ADDED] 1. spark.executor.instances
   Now: 7
   Reason: Optimal number of executors for parallel processing of 30 partitions. Provides balance between parallelism and overhead.

➕ [ADDED] 2. spark.executor.cores
   Now: 4
   Reason: Limiting cores per executor to prevent OOM when simultaneously loading multiple FAISS indexes into memory. Recommended ≤4 cores.

➕ [ADDED] 3. spark.executor.memory
   Now: 2g
   Reason: Executor memory calculated for loading FAISS indexes of size ~4.5 GB with overhead (1.5x). Prevents OOM during neighbor search.

➕ [ADDED] 4. spark.executor.memoryOverhead
   Now: 384m
   Reason: Memory overhead for JVM, serialization, and off-heap operations. 10% of executor memory, minimum 384 MB.

➕ [ADDED] 5. spark.sql.shuffle.partitions
   Now: 30
   Reason: Number of partitions for shuff

In [5]:
n_rows = 100_000_000
n_columns = 7
n_categorical = 0

# --- 3. Создание и использование калькулятора ---
calculator = SparkSessionCalculator(
    data_size_bytes=n_rows * n_columns * 8,  # Примерная оценка
    num_columns=n_columns,
    num_categorical_columns=n_categorical,
    # target_executor_cores=6,
    # target_executor_memory_gb=8
)
optimal_settings = calculator.calculate_optimal_settings()
print("\n Оптимальные настройки от калькулятора:")
print(f"  Executor instances: {optimal_settings.executor_instances}")
print(f"  Executor cores: {optimal_settings.executor_cores}")
print(f"  Executor memory: {optimal_settings.executor_memory}")
print(f"  Shuffle partitions: {optimal_settings.shuffle_partitions}")


 Оптимальные настройки от калькулятора:
  Executor instances: 8
  Executor cores: 4
  Executor memory: 2g
  Shuffle partitions: 35


In [4]:
import os
import time
from pyspark.sql import SparkSession
from hypex.utils.spark_config import SparkSessionCalculator

# --- 1. Настройки окружения для macOS (Важно!) ---
os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# Очистка старых сессий и переменных
try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        existing_spark.stop()
        print("✅ Существующая сессия остановлена.")
except:
    pass

for key in list(os.environ.keys()):
    if 'SPARK' in key or 'JAVA_OPTS' in key:
        del os.environ[key]

# --- 2. Конфигурация для калькулятора ---
NUM_EXECUTORS = 2
CORES_PER_EXECUTOR = 4
MEMORY_PER_EXECUTOR_MB = 2048

# Параметры для калькулятора (на основе вашего датасета)
n_rows = 10_000
n_columns = 7
n_categorical = 1

# --- 3. Создание и использование калькулятора ---
calculator = SparkSessionCalculator(
    data_size_bytes=n_rows * n_columns * 8,  # Примерная оценка
    num_columns=n_columns,
    num_categorical_columns=n_categorical,
    target_executor_cores=CORES_PER_EXECUTOR,
    target_executor_memory_gb=MEMORY_PER_EXECUTOR_MB / 1024
)

# Получаем оптимальные настройки
optimal_settings = calculator.calculate_optimal_settings()

print("\n📊 Оптимальные настройки от калькулятора:")
print(f"  Executor instances: {optimal_settings.executor_instances}")
print(f"  Executor cores: {optimal_settings.executor_cores}")
print(f"  Executor memory: {optimal_settings.executor_memory}")
print(f"  Shuffle partitions: {optimal_settings.shuffle_partitions}")

# --- 4. Создание Spark-сессии с настройками калькулятора ---
MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"
print(f"\n🚀 Запуск в режиме: {MASTER_URL}")

builder = (SparkSession.builder
    .master(MASTER_URL)
    .appName("LocalClusterTest")
    # Настройки драйвера
    .config("spark.driver.memory", "2g")
    # Настройки executor'ов из калькулятора
    .config("spark.executor.memory", f"{MEMORY_PER_EXECUTOR_MB}m")
    .config("spark.executor.cores", str(CORES_PER_EXECUTOR))
    .config("spark.executor.instances", str(NUM_EXECUTORS))
    # Дополнительные настройки из калькулятора
    .config("spark.memory.fraction", str(optimal_settings.memory_fraction))
    .config("spark.sql.shuffle.partitions", str(optimal_settings.shuffle_partitions))
    .config("spark.serializer", optimal_settings.serializer)
    .config("spark.sql.adaptive.enabled", "true")
)

sp_s = builder.getOrCreate()
sp_s.sparkContext.setLogLevel("WARN")

# --- 5. Проверка конфигурации ---
print(f"\n✅ Сессия создана.")
print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")
print(f"Shuffle Partitions: {sp_s.conf.get('spark.sql.shuffle.partitions')}")

# Проверка количества экзекуторов
time.sleep(3)
num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# --- 6. Тест на распределение ---
def print_executor_info(iterator):
    import os
    executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
    process_id = os.getpid()
    return [f"Executor ID: {executor_id}, PID: {process_id}"]

df = sp_s.range(0, 10, 1, 4)
result = df.rdd.mapPartitions(print_executor_info).collect()

print("\n🖥️ Где выполнялись задачи:")
for line in result:
    print(line)

# sp_s.stop()  # Раскомментируйте, если нужно остановить сессию


📊 Оптимальные настройки от калькулятора:
  Executor instances: 2
  Executor cores: 4
  Executor memory: 2g
  Shuffle partitions: 10

🚀 Запуск в режиме: local-cluster[2, 4, 2048]


26/08/14 13:33:47 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.75.216 instead (on interface wlo1)
26/08/14 13:33:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/14 13:33:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable



✅ Сессия создана.
Driver Memory Config: 2g
Executor Memory Config: 2048m
Shuffle Partitions: 10


📊 Активных экзекуторов (проверка через RDD): 2

🖥️ Где выполнялись задачи:
Executor ID: Driver/Local, PID: 527600
Executor ID: Driver/Local, PID: 527606
Executor ID: Driver/Local, PID: 527649
Executor ID: Driver/Local, PID: 527643


In [6]:
# # --- 1. Настройки окружения для macOS (Важно!) ---
# # На macOS иногда возникают проблемы с форком процессов Java (Executor'ы не стартуют).
# # Эта переменная часто решает проблему "Connection refused" или краши при запуске local-cluster
# os.environ['OBJC_DISABLE_INITIALIZE_FORK_SAFETY'] = 'YES'

# # Очистка старых сессий и переменных (как у вас было)
# try:
#     existing_spark = SparkSession.getActiveSession()
#     if existing_spark:
#         existing_spark.stop()
#         print("✅ Существующая сессия остановлена.")
# except:
#     pass

# for key in list(os.environ.keys()):
#     if 'SPARK' in key or 'JAVA_OPTS' in key:
#         del os.environ[key]

# # --- 2. Конфигурация Кластера ---
# # Формат: local-cluster[число_воркеров, ядер_на_воркер, память_на_воркер_в_МБ]
# # Мы просим 2 экзекутора, по 1 ядру, по 2 ГБ памяти каждый
# NUM_EXECUTORS = 2
# CORES_PER_EXECUTOR = 4
# MEMORY_PER_EXECUTOR_MB = 2048 

# MASTER_URL = f"local-cluster[{NUM_EXECUTORS}, {CORES_PER_EXECUTOR}, {MEMORY_PER_EXECUTOR_MB}]"

# print(f"🚀 Запуск в режиме: {MASTER_URL}")

# sp_s = (SparkSession.builder
#     .master(MASTER_URL)
#     .appName("LocalClusterTest")
#     # Память драйвера (остается у вас)
#     .config("spark.driver.memory", "2g") 
#     # Память экзекутора (должна соответствовать или быть меньше чем в master URL)
#     .config("spark.executor.memory", "2g")
#     .config("spark.executor.cores", "4")
#     .config("spark.executor.instances", NUM_EXECUTORS)
#     # Увеличиваем память под оверхед, чтобы избежать ошибок выделения памяти
#     .config("spark.memory.fraction", "0.6")
#     .config("spark.sql.shuffle.partitions", "4") # Для тестов меньше дефолтных 200
#     .getOrCreate()
# )

# sp_s.sparkContext.setLogLevel("WARN")

# # --- 3. Проверка конфигурации ---
# print(f"✅ Сессия создана.")
# print(f"Driver Memory Config: {sp_s.conf.get('spark.driver.memory')}")
# print(f"Executor Memory Config: {sp_s.conf.get('spark.executor.memory')}")

# # Проверка количества экзекуторов (может занять пару секунд на старт)
# import time
# time.sleep(3) 
# num_executors = len(sp_s.sparkContext.parallelize(range(10), NUM_EXECUTORS).glom().collect())
# print(f"📊 Активных экзекуторов (проверка через RDD): {num_executors}")

# # --- 4. Тест на распределение (Пример) ---
# # Чтобы убедиться, что задача ушла на экзекуторы, а не осталась на драйвере
# def print_executor_info(iterator):
#     import os
#     # Получаем ID экзекутора из переменных окружения процесса
#     executor_id = os.environ.get('SPARK_EXECUTOR_ID', 'Driver/Local')
#     process_id = os.getpid()
#     return [f"Executor ID: {executor_id}, PID: {process_id}"]

# # Создаем датафрейм и применяем трансформацию
# df = sp_s.range(0, 10, 1, 4) # 4 партиции
# result = df.rdd.mapPartitions(print_executor_info).collect()

# print("\n🖥️ Где выполнялись задачи:")
# for line in result:
#     print(line)

# # Не забывайте останавливать сессию в конце скрипта, так как процессы тяжелые
# # sp_s.stop() 

In [5]:
n_rows = 10000
n = n_rows
df = pd.DataFrame({
    "treatment": np.random.choice([0, 1], size=n, p=[0.6, 0.4]),
    "feat_num_1": np.random.normal(loc=10, scale=3, size=n),
    "feat_num_2": np.random.normal(loc=-2, scale=1.5, size=n),
    "feat_cat": np.random.choice(["A", "B", "C"], size=n),
    "target": np.random.normal(loc=100, scale=10, size=n)
})

In [4]:
nn = 200
index_df = pd.DataFrame(
    {
        '0': np.random.randint(0, 50, nn),
        '1': np.random.randint(0, 50, nn),
        '2': np.random.randint(0, 50, nn),
        '3': np.random.randint(0, 50, nn),
        '4': np.random.randint(0, 50, nn),
        'group': [0] * (nn//4) + [1] * (nn//4) + [2] * (nn//4) + [3] * (nn//4)
    }
)

index_df = pd.concat([index_df, pd.DataFrame(data={
    '0': np.nan,
    '1': np.nan,
    '2': np.nan,
    '3': np.nan,
    '4': np.nan,
    'group': np.nan
}, index=[0])]).reset_index(drop=True)
index_df.head(10)

,0,1,2,3,4,group
0,40.0,48.0,48.0,43.0,14.0,0.0
1,2.0,33.0,6.0,22.0,40.0,0.0
2,35.0,32.0,28.0,2.0,37.0,0.0
3,8.0,10.0,23.0,29.0,29.0,0.0
4,47.0,19.0,39.0,11.0,27.0,0.0
5,18.0,39.0,47.0,28.0,49.0,0.0
6,24.0,40.0,7.0,18.0,29.0,0.0
7,29.0,30.0,15.0,13.0,2.0,0.0
8,8.0,9.0,0.0,41.0,14.0,0.0
9,15.0,36.0,46.0,41.0,40.0,0.0


In [5]:
session = (
            SparkSession.builder
            .master("local[*]")
            .config("spark.driver.memory", "4g")
            .config("spark.executor.memory", "4g")
            .config("spark.memory.fraction", "0.8") 
            .config("spark.memory.storageFraction", "0.3")
            # .config("spark.jars.packages", "ch.cern.sparkmeasure:spark-measure_2.12:0.23") 
            .getOrCreate()
          )

26/08/05 12:50:06 WARN Utils: Your hostname, eric-Katana-17-B12UCR resolves to a loopback address: 127.0.1.1; using 10.240.73.68 instead (on interface wlo1)
26/08/05 12:50:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/05 12:50:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    data=session.createDataFrame(index_df),
    # data=sp_s.createDataFrame(index_df),
    session=session
    # session=sp_s
)

index_ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all d

,0,1,2,3,4,group
0,40.0,48.0,48.0,43.0,14.0,0.0
1,2.0,33.0,6.0,22.0,40.0,0.0
2,35.0,32.0,28.0,2.0,37.0,0.0
3,8.0,10.0,23.0,29.0,29.0,0.0
4,47.0,19.0,39.0,11.0,27.0,0.0
...,...,...,...,...,...,...
196,22.0,34.0,33.0,36.0,12.0,3.0
197,34.0,44.0,28.0,2.0,29.0,3.0
198,0.0,23.0,33.0,33.0,13.0,3.0
199,4.0,22.0,15.0,2.0,34.0,3.0


In [15]:
"""
PANDAS case
"""
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

pandas_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         ),
        #         Chi2Test(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole()
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
pandas_result = pandas_experiment.execute(ExperimentData(dataset))

2026-08-05 12:53:10 | INFO     | hypex.experiment | ============================================================
2026-08-05 12:53:10 | INFO     | hypex.experiment | Spark Session Info:
2026-08-05 12:53:10 | INFO     | hypex.experiment |   Master: local-cluster[2, 4, 2048]
2026-08-05 12:53:10 | INFO     | hypex.experiment |   App name: LocalClusterTest
2026-08-05 12:53:10 | INFO     | hypex.experiment |   Driver memory: 2g
2026-08-05 12:53:10 | INFO     | hypex.experiment |   Executor memory: 2048m
2026-08-05 12:53:10 | INFO     | hypex.experiment |   Executor cores: 4
2026-08-05 12:53:10 | INFO     | hypex.experiment |   Executor instances: 8
2026-08-05 12:53:10 | INFO     | hypex.experiment |   Spark version: 3.5.1
2026-08-05 12:53:10 | INFO     | hypex.experiment | ============================================================
2026-08-05 12:53:10 | INFO     | hypex.experiment | ▶ Process started: DummyEncoder [pandas]
2026-08-05 12:53:10 | INFO     | hypex.experiment | ✓ Process finish

In [ ]:
pandas_result.ds

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_matched_target
0,0,14.254047,-2.106962,A,82.212611,0.0,0.0,5341,467,0.001729,105.747777
1,0,13.379692,-0.090667,A,91.196115,0.0,0.0,4932,4916,0.023358,91.616126
2,0,12.349636,-2.187006,A,100.28066,0.0,0.0,8826,2847,-0.003827,102.44747
3,0,7.14077,-3.882448,A,102.332887,0.0,0.0,1328,9743,-0.000127,94.694204
4,0,4.455735,-2.970478,B,110.80817,1.0,0.0,6188,4260,-0.018215,87.785081
...,...,...,...,...,...,...,...,...,...,...,...
9995,0,12.755274,-2.045584,B,104.012952,1.0,0.0,618,5052,0.047694,89.106389
9996,0,7.250438,-4.576099,A,93.854306,0.0,0.0,3372,8244,0.0349,84.563756
9997,1,5.096665,-1.409165,C,98.762038,0.0,1.0,6479,7203,-0.032802,99.620516
9998,1,8.498396,-5.162666,B,87.142353,1.0,0.0,8001,3915,0.020642,103.063151


In [ ]:
pandas_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.030577        0.267131  0.908870 -0.554154  0.493000
 ATC     0.018248        0.358927  0.959452 -0.685249  0.721746
 ATE    -0.001379        0.267769  0.995890 -0.526206  0.523447
 
 3 rows × 5 columns}

In [ ]:
# Например: 10_000_000 строк × 8 колонок × 8 байт = 640_000_000 байт (~0.6 ГБ)
data_size_bytes = 10_000_000 * 8 * 8  # если известно
calculator = SparkSessionCalculator(
    data_size_bytes=data_size_bytes,
    num_columns=8,
    num_categorical_columns=1
)

optimal_settings = calculator.calculate_optimal_settings()
current = calculator.check_current_settings(sp_s)
calculator.generate_recommendations(current, optimal_settings)
calculator.print_recommendations()


РЕКОМЕНДАЦИИ ПО НАСТРОЙКЕ SPARK-СЕССИИ

🟠 [HIGH] spark.sql.shuffle.partitions
   Текущее значение: 4
   Рекомендуемое значение: 10
   Причина: Оптимизируйте количество партиций для баланса между размером индекса и параллелизмом

🟡 [MEDIUM] spark.serializer
   Текущее значение: unknown
   Рекомендуемое значение: org.apache.spark.serializer.KryoSerializer
   Причина: Используйте KryoSerializer для более эффективной сериализации



In [ ]:
from hypex.utils.spark_config import SparkSessionCalculator, SparkSettings

# Вариант 1: Создание новой сессии с оптимальными настройками
calculator = SparkSessionCalculator(
    data_size_bytes=10_000_000_000,  # 10 GB
    num_columns=8,
    num_categorical_columns=2,
    target_executor_cores=4,
    target_executor_memory_gb=4.0
)

settings = calculator.calculate_optimal_settings()
spark = calculator.create_optimal_session(settings)

# Вариант 2: Проверка и оптимизация существующей сессии
spark = SparkSession.builder.getOrCreate()
calculator = SparkSessionCalculator(num_rows=1_000_000, num_columns=8)

current = calculator.check_current_settings(spark)
optimal = calculator.calculate_optimal_settings()
calculator.generate_recommendations(current, optimal)
calculator.print_recommendations()
calculator.apply_settings(spark, optimal)

In [6]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

spark_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=10,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=10,
        ),
        MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         ),
        #         Chi2Test(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole()
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
spark_result = spark_experiment.execute(
    ExperimentData(dataset)
)

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
2026-08-14 13:33:56 | INFO     | hypex.experiment | ============================================================
2026-08-14 13:33:56 | INFO     | hypex.experiment | Spark Session Info:
2026-08-14 13:33:56 | INFO     | hypex.experiment |   Master: local-cluster[2, 4, 2048]
2026-08-14 13:33:56 | INFO     | hypex.experiment |   App name: LocalClusterTest
2026-08-14 13:33:56 | INFO     | hypex.experiment |   Driver memory: 2g
2026-08-14 13:33:56 | INFO     | hypex.experiment |   Executor memory: 2048m
2026-08-14 13:33:56 | INFO     | hypex.experiment |   Executor cores: 4
2026-08-14 13:33:56 | INFO     | hypex.experiment |   Executor instances: 8
2026-08-14 13:33:56 | INFO     | hypex.experiment |   Spark version: 3.5.1


In [7]:
from hypex.utils import FaissIndexStorage

FaissIndexStorage.cleanup()

In [ ]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,FaissNearestNeighbors┴┴┴2,FaissNearestNeighbors┴┴┴3,FaissNearestNeighbors┴┴┴4,FaissNearestNeighbors┴┴┴5,FaissNearestNeighbors┴┴┴6,FaissNearestNeighbors┴┴┴7,FaissNearestNeighbors┴┴┴8,FaissNearestNeighbors┴┴┴9
5128,0,13.416287,-3.553226,C,99.348902,0.0,1.0,3097,505,2777,4604,8180,5535,3914,9684,4797,5326
5131,0,10.393949,0.433733,B,94.363742,1.0,0.0,7425,8,5863,5878,8387,7901,3755,9667,6317,4092
5154,0,8.810371,-1.160441,B,105.752635,1.0,0.0,6015,6535,5348,6544,6131,7492,4274,4759,9883,3403
5165,0,9.761503,-3.081742,A,87.956967,0.0,0.0,8279,4820,3384,5057,6617,4240,9995,8538,7695,5414
5173,0,8.953117,-3.165956,C,88.330607,0.0,1.0,7052,6333,5092,4419,638,2527,8415,5066,9785,6615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1,9.778388,-0.813486,C,108.117268,0.0,1.0,1806,1912,1644,9663,3382,4583,5184,7271,9281,9433
9996,1,13.110116,-1.720605,B,90.96973,1.0,0.0,4793,6542,2135,1871,4110,9213,1005,5344,9098,4118
9997,1,3.417471,1.050526,C,114.424125,0.0,1.0,2257,3669,807,2483,2443,3469,9784,3124,5806,5947
9998,0,10.113562,-4.595439,A,104.531089,0.0,0.0,9385,6314,9864,7413,2186,8110,2231,5337,7793,812


26/07/14 11:20:48 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 62798772 ms exceeds timeout 120000 ms
26/07/14 11:20:48 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 62797263 ms exceeds timeout 120000 ms
26/07/14 11:20:48 WARN Master: Removing worker-20260713154124-10.240.76.27-43443 because we got no heartbeat in 60 seconds
26/07/14 11:20:48 WARN Master: Removing worker-20260713154124-10.240.76.27-45653 because we got no heartbeat in 60 seconds
26/07/14 11:20:48 WARN Master: App app-20260713154124-0000 requires more resource than any of Workers could have.
26/07/14 11:22:48 ERROR Utils: Uncaught exception in thread kill-executor-thread
org.apache.spark.rpc.RpcTimeoutException: Futures timed out after [120 seconds]. This timeout is controlled by spark.rpc.askTimeout
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIf

In [ ]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.368218        0.656622  0.574950 -1.655197  0.918762
 ATC     0.000346        0.542405  0.999492 -1.062768  1.063459
 ATE    -0.169889        0.519928  0.743853 -1.188949  0.849170
 
 3 rows × 5 columns}

In [ ]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_matched_target
2,1,14.208973,-1.194687,B,93.144985,1.0,0.0,2769,2005,-1.850296,94.99792
4,0,9.021018,-1.372699,C,104.130018,0.0,1.0,1205,2431,15.591174,88.517108
5,0,9.043075,-0.657164,C,89.267936,0.0,1.0,496,4435,-22.602262,111.901704
8,1,5.938445,-1.210908,A,107.483695,0.0,0.0,4927,545,10.747533,96.720872
12,0,11.028771,-1.540009,C,84.923102,0.0,1.0,2297,4806,-19.426577,104.376759
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,9.562625,-4.569026,C,82.533743,0.0,1.0,577,4044,-21.166491,103.729759
4996,1,11.728486,-1.896468,B,106.871564,1.0,0.0,672,3066,11.12951,95.726223
4997,1,10.200868,-2.082435,B,92.33909,1.0,0.0,3122,982,-0.148631,92.48793
4998,0,4.210678,-0.564392,A,95.937813,0.0,0.0,1896,2221,-1.605121,97.545157


In [ ]:
spark_result.ds.unpersist()

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_target_matched,MatchingMetrics┴┴
2,0,12.045232,-3.791779,B,96.912328,1.0,0.0,3451,4010,-1.752359,98.667182,98.667182
4,0,7.5948,-3.506998,C,97.817065,0.0,1.0,662,639,-0.725902,98.54402,98.54402
5,0,11.261949,-2.80301,C,83.018658,0.0,1.0,3132,3036,-17.915917,100.960448,100.960448
8,1,11.935777,-1.693673,C,107.441409,0.0,1.0,4609,1616,13.27009,94.152028,94.152028
12,0,16.536284,-4.516249,B,96.54423,1.0,0.0,2819,3982,-6.57412,103.128048,103.128048
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1,13.395537,-2.90735,B,92.776833,1.0,0.0,2244,3676,-5.429742,98.214405,98.214405
4996,0,12.690551,1.248961,B,105.992565,1.0,0.0,3858,2290,7.362271,98.619517,98.619517
4997,1,14.991156,-0.602245,C,99.631197,0.0,1.0,2338,183,10.378529,89.237694,89.237694
4998,1,8.153213,-0.34911,B,111.509293,1.0,0.0,3763,4132,10.817036,100.676567,100.676567


In [ ]:
spark_result.variables["MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||_feat_cat_B', 'DummyEncoder||_feat_cat_C']"]["['feat_num_1', 'feat_num_2', 'DummyEncoder┴┴_feat_cat_B', 'DummyEncoder┴┴_feat_cat_C']"]

,feat_num_1,feat_num_2,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C
feat_num_1,0.329423,0.005064,-0.002166,-0.001165
feat_num_2,0.000000,0.674589,0.004112,0.004982
DummyEncoder┴┴_feat_cat_B,0.000000,0.000000,2.103345,1.198875
DummyEncoder┴┴_feat_cat_C,0.000000,0.000000,0.000000,2.456418


In [ ]:
spark_result.field_search(AdditionalStatisticRole())

[]

In [ ]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,"Bias┴┴['target', 'target_matched']",MatchingMetrics┴┴
2507,1,9.344322,0.061681,C,122.704119,0.0,1.0,2593,3491,0.001844,87.343549
2509,1,15.051634,-1.646648,C,103.488801,0.0,1.0,1057,2774,0.007622,109.718559
2513,0,6.778776,-3.909684,B,99.504827,1.0,0.0,4593,1582,0.001345,104.010242
2529,1,10.611894,-1.060253,B,101.451064,1.0,0.0,1125,4441,-0.006783,92.709322
2532,0,13.309939,-3.323301,A,98.405159,0.0,0.0,4522,3631,0.007523,87.343303
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,4.215119,-1.728702,A,116.791128,0.0,0.0,1437,3900,0.022995,93.282247
4996,0,9.728035,-1.025706,C,104.852101,0.0,1.0,597,463,0.022611,100.832959
4997,1,5.080752,-2.045856,C,94.247351,0.0,1.0,3223,2161,-0.027136,88.786346
4998,0,10.767351,-0.634082,A,88.233454,0.0,0.0,4690,3723,0.006813,110.698601


In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [ ]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error  P-value  CI Lower  CI Upper
 ATT    -0.386312             0.0      0.0 -0.386312 -0.386312
 ATC     0.223290             0.0      0.0  0.223290  0.223290
 ATE    -0.024031             0.0      0.0 -0.024031 -0.024031
 
 3 rows × 5 columns}

In [ ]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [ ]:
sp_s.stop()

In [12]:
index_ds = Dataset(
    roles={
        'group': TargetRole()
    },
    # data=index_df
    data=session.createDataFrame(index_df),
    # data=sp_s.createDataFrame(index_df),
    session=session
    # session=sp_s
)

index_ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all d

,0,1,2,3,4,group
0,40,23,10,23,15,0
1,20,36,41,43,49,0
...,...,...,...,...,...,...
198,16,14,49,11,6,3
199,16,42,36,35,15,3


In [ ]:
"""
PANDAS case
"""
# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=df,
    backend=BackendsEnum.pandas,
)

pandas_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=2,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        MatchingMetrics(
                grouping_role=TreatmentRole(),
                target_roles=[TargetRole()],
                metric="ate",
                n_neighbors=2,
        ),
        MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         ),
        #         Chi2Test(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole()
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
pandas_result = pandas_experiment.execute(ExperimentData(dataset))

2026-07-17 15:24:34 | INFO     | hypex.experiment | ▶ Process started: DummyEncoder [pandas]
2026-07-17 15:24:34 | INFO     | hypex.experiment | ✓ Process finished: DummyEncoder in 0.017s
2026-07-17 15:24:34 | INFO     | hypex.experiment | ▶ Process started: MahalanobisDistance [pandas]


/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  return int(self.data[group_cols].nunique())
2026-07-17 15:24:34 | INFO     | hypex.experiment | ✓ Process finished: MahalanobisDistance in 0.060s
2026-07-17 15:24:34 | INFO     | hypex.experiment | ▶ Process started: TypeCaster [pandas]
2026-07-17 15:24:34 | INFO     | hypex.experiment | ✓ Process finished: TypeCaster in 0.010s
2026-07-17 15:24:34 | INFO     | hypex.experiment | ▶ Process started: FaissNearestNeighbors [pandas]
/home/eric/HypEx/HypEx/hypex/dataset/backends/pandas_backend.py:1175: FutureWarning: Calling int on a 

In [6]:
pandas_result.ds

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_matched_target
0,0,14.254047,-2.106962,A,82.212611,0.0,0.0,5341,467,0.001729,105.747777
1,0,13.379692,-0.090667,A,91.196115,0.0,0.0,4932,4916,0.023358,91.616126
2,0,12.349636,-2.187006,A,100.28066,0.0,0.0,8826,2847,-0.003827,102.44747
3,0,7.14077,-3.882448,A,102.332887,0.0,0.0,1328,9743,-0.000127,94.694204
4,0,4.455735,-2.970478,B,110.80817,1.0,0.0,6188,4260,-0.018215,87.785081
...,...,...,...,...,...,...,...,...,...,...,...
9995,0,12.755274,-2.045584,B,104.012952,1.0,0.0,618,5052,0.047694,89.106389
9996,0,7.250438,-4.576099,A,93.854306,0.0,0.0,3372,8244,0.0349,84.563756
9997,1,5.096665,-1.409165,C,98.762038,0.0,1.0,6479,7203,-0.032802,99.620516
9998,1,8.498396,-5.162666,B,87.142353,1.0,0.0,8001,3915,0.020642,103.063151


In [7]:
pandas_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.030577        0.267131  0.908870 -0.554154  0.493000
 ATC     0.018248        0.358927  0.959452 -0.685249  0.721746
 ATE    -0.001379        0.267769  0.995890 -0.526206  0.523447
 
 3 rows × 5 columns}


ЛОГ ОПТИМИЗАЦИИ КОНФИГА SPARK-СЕССИИ

📊 ИТОГО ИЗМЕНЕНИЙ: 13
   ➕ Добавлено: 10
   🔄 Изменено: 3

--------------------------------------------------------------------------------

🔄 [ИЗМЕНЕНО] 1. spark.executor.instances
   Было: 2
   Стало: 15
   Причина: Оптимальное количество executor'ов для параллельной обработки 63 партиций. Обеспечивает баланс между параллелизмом и накладными расходами.

🔄 [ИЗМЕНЕНО] 2. spark.executor.cores
   Было: 8
   Стало: 4
   Причина: Ограничение ядер на executor для предотвращения OOM при одновременной загрузке нескольких индексов FAISS в память. Рекомендуется ≤4 ядра.

➕ [ДОБАВЛЕНО] 3. spark.executor.memoryOverhead
   Стало: 384m
   Причина: Overhead памяти для JVM, сериализации и off-heap операций. 10% от executor memory, минимум 384 MB.

🔄 [ИЗМЕНЕНО] 4. spark.sql.shuffle.partitions
   Было: 4
   Стало: 63
   Причина: Количество партиций для shuffle-операций. Оптимизировано для размера данных 9.3 GB, обеспечивает баланс между параллелизмом и размером ин

In [11]:
# Например: 10_000_000 строк × 8 колонок × 8 байт = 640_000_000 байт (~0.6 ГБ)
data_size_bytes = 10_000_000 * 8 * 8  # если известно
calculator = SparkSessionCalculator(
    data_size_bytes=data_size_bytes,
    num_columns=8,
    num_categorical_columns=1
)

optimal_settings = calculator.calculate_optimal_settings()
current = calculator.check_current_settings(sp_s)
calculator.generate_recommendations(current, optimal_settings)
calculator.print_recommendations()


РЕКОМЕНДАЦИИ ПО НАСТРОЙКЕ SPARK-СЕССИИ

🟠 [HIGH] spark.sql.shuffle.partitions
   Текущее значение: 4
   Рекомендуемое значение: 10
   Причина: Оптимизируйте количество партиций для баланса между размером индекса и параллелизмом

🟡 [MEDIUM] spark.serializer
   Текущее значение: unknown
   Рекомендуемое значение: org.apache.spark.serializer.KryoSerializer
   Причина: Используйте KryoSerializer для более эффективной сериализации



In [ ]:
from hypex.utils.spark_config import SparkSessionCalculator, SparkSettings

# Вариант 1: Создание новой сессии с оптимальными настройками
calculator = SparkSessionCalculator(
    data_size_bytes=10_000_000_000,  # 10 GB
    num_columns=8,
    num_categorical_columns=2,
    target_executor_cores=4,
    target_executor_memory_gb=4.0
)

settings = calculator.calculate_optimal_settings()
spark = calculator.create_optimal_session(settings)

# Вариант 2: Проверка и оптимизация существующей сессии
spark = SparkSession.builder.getOrCreate()
calculator = SparkSessionCalculator(num_rows=1_000_000, num_columns=8)

current = calculator.check_current_settings(spark)
optimal = calculator.calculate_optimal_settings()
calculator.generate_recommendations(current, optimal)
calculator.print_recommendations()
calculator.apply_settings(spark, optimal)

In [6]:
"""
PYSPARK case
"""
# 3. Конвертация в Spark + ОБЯЗАТЕЛЬНАЯ колонка `index` (требование Faiss)
spark_df = sp_s.createDataFrame(df)

# 4. Обёртка в Dataset фреймворка
roles = {
    "treatment": TreatmentRole(),
    "feat_num_1": FeatureRole(),
    "feat_num_2": FeatureRole(),
    "feat_cat": FeatureRole(str),
    "target": TargetRole(),
    # "index": FeatureRole()  # индекс тоже должен быть в ролях, чтобы не отфильтровался
}

dataset = Dataset(
    roles=roles,
    data=spark_df,
    # data=session.createDataFrame(df),
    # data=df,
    backend=BackendsEnum.spark,
    session=sp_s,
    # session=session
)

spark_experiment = Experiment(
    executors=[
        DummyEncoder(),
        MahalanobisDistance(
            grouping_role=TreatmentRole(),
            weights=None
        ),
        TypeCaster(
            dtype={int: float},
            roles=[FeatureRole(), TargetRole()],
        ),
        FaissNearestNeighbors(
            grouping_role=TreatmentRole(),
            two_sides=True,
            test_pairs=False,
            faiss_mode="auto",
            n_neighbors=10,
        ),
        Bias(
            grouping_role=TreatmentRole(), 
            target_roles=[TargetRole()]
        ),
        # MatchingMetrics(
        #         grouping_role=TreatmentRole(),
        #         target_roles=[TargetRole()],
        #         metric="ate",
        #         n_neighbors=10,
        # ),
        # MatchingAnalyzer(),
        # OnRoleExperiment(
        #     executors=[
        #         TTest(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole(),
        #         ),
        #         Chi2Test(
        #             grouping_role=TreatmentRole(),
        #             compare_by="matched_pairs",
        #             baseline_role=AdditionalMatchingRole()
        #         )
        #     ],
        #     role=FeatureRole()
        # )
    ]
)
spark_result = spark_experiment.execute(
    ExperimentData(dataset)
)

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
2026-08-04 10:54:05 | INFO     | hypex.experiment | ============================================================
2026-08-04 10:54:05 | INFO     | hypex.experiment | Spark Session Info:
2026-08-04 10:54:05 | INFO     | hypex.experiment |   Master: local-cluster[2, 4, 2048]
2026-08-04 10:54:05 | INFO     | hypex.experiment |   App name: LocalClusterTest
2026-08-04 10:54:05 | INFO     | hypex.experiment |   Driver memory: 2g
2026-08-04 10:54:05 | INFO     | hypex.experiment |   Executor memory: 2048m
2026-08-04 10:54:05 | INFO     | hypex.experiment |   Executor cores: 4
2026-08-04 10:54:05 | INFO     | hypex.experiment |   Executor instances: 8
2026-08-04 10:54:05 | INFO     | hypex.experiment |   Spark version: 3.5.1


In [7]:
from hypex.utils import FaissIndexStorage

FaissIndexStorage.cleanup()

Вот одно из предложений для модификации т-теста для двух выборок. Верно ли оно?
Рассматривая статистику:
$$
\text{t-stat} =
 \cfrac{\overset{-}{Z} - \overset{-}{Y}}{\sqrt{\cfrac{\sigma^2_1}{n}} + \cfrac{\sigma^2_2}{m}}
$$

где $\overset{-}{Z} = \cfrac{1}{n}\sum_{i=1}^n Z_i$ и $\overset{-}{Y} = \cfrac{1}{m} \sum_{j=1}^m Y_j$

Либо:

$$
\text{new t-stat} =
\cfrac{1}{\sqrt{\sigma^2_1 + \gamma^2 \sigma^2_2}} \sum_{i=1}^n (Z_i - E [Z_i])u_i
-\cfrac{\gamma}{\sqrt{\sigma^2_1 + \gamma^2 \sigma^2_2}} \sum_{j=1}^m (Y_j - E [Y_j])v_j
$$

С подходящими векторами $||u|| = 1$ и $||v|| = 1$, $u \in R^n, v \in R^m$ и параметром 
$$
\gamma = \cfrac{\sum_{i=1}^n u_i}{\sum_{j=1}^m v_j}
$$

In [ ]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,FaissNearestNeighbors┴┴┴2,FaissNearestNeighbors┴┴┴3,FaissNearestNeighbors┴┴┴4,FaissNearestNeighbors┴┴┴5,FaissNearestNeighbors┴┴┴6,FaissNearestNeighbors┴┴┴7,FaissNearestNeighbors┴┴┴8,FaissNearestNeighbors┴┴┴9
5128,0,13.416287,-3.553226,C,99.348902,0.0,1.0,3097,505,2777,4604,8180,5535,3914,9684,4797,5326
5131,0,10.393949,0.433733,B,94.363742,1.0,0.0,7425,8,5863,5878,8387,7901,3755,9667,6317,4092
5154,0,8.810371,-1.160441,B,105.752635,1.0,0.0,6015,6535,5348,6544,6131,7492,4274,4759,9883,3403
5165,0,9.761503,-3.081742,A,87.956967,0.0,0.0,8279,4820,3384,5057,6617,4240,9995,8538,7695,5414
5173,0,8.953117,-3.165956,C,88.330607,0.0,1.0,7052,6333,5092,4419,638,2527,8415,5066,9785,6615
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,1,9.778388,-0.813486,C,108.117268,0.0,1.0,1806,1912,1644,9663,3382,4583,5184,7271,9281,9433
9996,1,13.110116,-1.720605,B,90.96973,1.0,0.0,4793,6542,2135,1871,4110,9213,1005,5344,9098,4118
9997,1,3.417471,1.050526,C,114.424125,0.0,1.0,2257,3669,807,2483,2443,3469,9784,3124,5806,5947
9998,0,10.113562,-4.595439,A,104.531089,0.0,0.0,9385,6314,9864,7413,2186,8110,2231,5337,7793,812


26/07/14 11:20:48 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 62798772 ms exceeds timeout 120000 ms
26/07/14 11:20:48 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 62797263 ms exceeds timeout 120000 ms
26/07/14 11:20:48 WARN Master: Removing worker-20260713154124-10.240.76.27-43443 because we got no heartbeat in 60 seconds
26/07/14 11:20:48 WARN Master: Removing worker-20260713154124-10.240.76.27-45653 because we got no heartbeat in 60 seconds
26/07/14 11:20:48 WARN Master: App app-20260713154124-0000 requires more resource than any of Workers could have.
26/07/14 11:22:48 ERROR Utils: Uncaught exception in thread kill-executor-thread
org.apache.spark.rpc.RpcTimeoutException: Futures timed out after [120 seconds]. This timeout is controlled by spark.rpc.askTimeout
	at org.apache.spark.rpc.RpcTimeout.org$apache$spark$rpc$RpcTimeout$$createRpcTimeoutException(RpcTimeout.scala:47)
	at org.apache.spark.rpc.RpcTimeout$$anonfun$addMessageIf

In [9]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error   P-value  CI Lower  CI Upper
 ATT    -0.368218        0.656622  0.574950 -1.655197  0.918762
 ATC     0.000346        0.542405  0.999492 -1.062768  1.063459
 ATE    -0.169889        0.519928  0.743853 -1.188949  0.849170
 
 3 rows × 5 columns}

In [7]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_matched_target
2,1,14.208973,-1.194687,B,93.144985,1.0,0.0,2769,2005,-1.850296,94.99792
4,0,9.021018,-1.372699,C,104.130018,0.0,1.0,1205,2431,15.591174,88.517108
5,0,9.043075,-0.657164,C,89.267936,0.0,1.0,496,4435,-22.602262,111.901704
8,1,5.938445,-1.210908,A,107.483695,0.0,0.0,4927,545,10.747533,96.720872
12,0,11.028771,-1.540009,C,84.923102,0.0,1.0,2297,4806,-19.426577,104.376759
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,9.562625,-4.569026,C,82.533743,0.0,1.0,577,4044,-21.166491,103.729759
4996,1,11.728486,-1.896468,B,106.871564,1.0,0.0,672,3066,11.12951,95.726223
4997,1,10.200868,-2.082435,B,92.33909,1.0,0.0,3122,982,-0.148631,92.48793
4998,0,4.210678,-0.564392,A,95.937813,0.0,0.0,1896,2221,-1.605121,97.545157


In [19]:
spark_result.ds.unpersist()

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,Bias┴┴target_bias,Bias┴┴target_target_matched,MatchingMetrics┴┴
2,0,12.045232,-3.791779,B,96.912328,1.0,0.0,3451,4010,-1.752359,98.667182,98.667182
4,0,7.5948,-3.506998,C,97.817065,0.0,1.0,662,639,-0.725902,98.54402,98.54402
5,0,11.261949,-2.80301,C,83.018658,0.0,1.0,3132,3036,-17.915917,100.960448,100.960448
8,1,11.935777,-1.693673,C,107.441409,0.0,1.0,4609,1616,13.27009,94.152028,94.152028
12,0,16.536284,-4.516249,B,96.54423,1.0,0.0,2819,3982,-6.57412,103.128048,103.128048
...,...,...,...,...,...,...,...,...,...,...,...,...
4995,1,13.395537,-2.90735,B,92.776833,1.0,0.0,2244,3676,-5.429742,98.214405,98.214405
4996,0,12.690551,1.248961,B,105.992565,1.0,0.0,3858,2290,7.362271,98.619517,98.619517
4997,1,14.991156,-0.602245,C,99.631197,0.0,1.0,2338,183,10.378529,89.237694,89.237694
4998,1,8.153213,-0.34911,B,111.509293,1.0,0.0,3763,4132,10.817036,100.676567,100.676567


In [12]:
spark_result.variables["MahalanobisDistance┴┴['feat_num_1', 'feat_num_2', 'DummyEncoder||_feat_cat_B', 'DummyEncoder||_feat_cat_C']"]["['feat_num_1', 'feat_num_2', 'DummyEncoder┴┴_feat_cat_B', 'DummyEncoder┴┴_feat_cat_C']"]

,feat_num_1,feat_num_2,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C
feat_num_1,0.329423,0.005064,-0.002166,-0.001165
feat_num_2,0.000000,0.674589,0.004112,0.004982
DummyEncoder┴┴_feat_cat_B,0.000000,0.000000,2.103345,1.198875
DummyEncoder┴┴_feat_cat_C,0.000000,0.000000,0.000000,2.456418


In [15]:
spark_result.field_search(AdditionalStatisticRole())

[]

In [7]:
spark_result.ds

/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: `to_pandas` loads all data into the driver's memory. It should only be used if the resulting pandas DataFrame is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/home/eric/.local/lib/python3.8/site-packages/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning:

,treatment,feat_num_1,feat_num_2,feat_cat,target,DummyEncoder┴┴_feat_cat_B,DummyEncoder┴┴_feat_cat_C,FaissNearestNeighbors┴┴┴0,FaissNearestNeighbors┴┴┴1,"Bias┴┴['target', 'target_matched']",MatchingMetrics┴┴
2507,1,9.344322,0.061681,C,122.704119,0.0,1.0,2593,3491,0.001844,87.343549
2509,1,15.051634,-1.646648,C,103.488801,0.0,1.0,1057,2774,0.007622,109.718559
2513,0,6.778776,-3.909684,B,99.504827,1.0,0.0,4593,1582,0.001345,104.010242
2529,1,10.611894,-1.060253,B,101.451064,1.0,0.0,1125,4441,-0.006783,92.709322
2532,0,13.309939,-3.323301,A,98.405159,0.0,0.0,4522,3631,0.007523,87.343303
...,...,...,...,...,...,...,...,...,...,...,...
4995,0,4.215119,-1.728702,A,116.791128,0.0,0.0,1437,3900,0.022995,93.282247
4996,0,9.728035,-1.025706,C,104.852101,0.0,1.0,597,463,0.022611,100.832959
4997,1,5.080752,-2.045856,C,94.247351,0.0,1.0,3223,2161,-0.027136,88.786346
4998,0,10.767351,-0.634082,A,88.233454,0.0,0.0,4690,3723,0.006813,110.698601


In [ ]:
# for label, ds in result.variables['Bias┴┴[\'target\', \'target_matched\']'].items():
#     ds.to_small_dataset().data.to_csv(f"{label}.csv")

In [6]:
spark_result.analysis_tables

{'MatchingAnalyzer┴┴':      Effect Size  Standard Error  P-value  CI Lower  CI Upper
 ATT    -0.386312             0.0      0.0 -0.386312 -0.386312
 ATC     0.223290             0.0      0.0  0.223290  0.223290
 ATE    -0.024031             0.0      0.0 -0.024031 -0.024031
 
 3 rows × 5 columns}

In [9]:
sp_s.stop()

In [ ]:
r = pd.read_csv('result_[5].csv', names=['index', 'value'])
r.dropna(subset=['index']).fillna(0)

In [ ]:

# 5. Настройка Matching
matching = Matching(
    distance="mahalanobis",
    # metric="ate",
    bias_estimation=False,      # отключаем для упрощения дебага
    quality_tests=["t-test"],   # только t-test для скорости
    faiss_mode="base",          # "base" → IndexFlatL2 (без IVF), проще отлаживать
    n_neighbors=1,
    encode_categories=True      # DummyEncoder включится автоматически
)

# 6. Запуск
print("🚀 Запуск пайплайна Matching...")
result_data = matching.execute(dataset)

print("✅ Выполнено успешно!")
print(f"📊 Additional Fields: {result_data.additional_fields.columns}")
print(f"📦 Groups Keys: {list(result_data.groups.keys())}")

In [8]:
sp_s.stop()